# LSTM-Based Stock Market Prediction - Model Reproduction (Part B)

## Overview
This notebook reproduces the LSTM-based cross-sectional stock prediction framework from Fischer & Krauss (2017).
We train four models on S&P 500 constituent stocks from 1992-2015 across 23 rolling study periods.

## Model Architecture
- **LSTM**: 1 input feature × 240 timesteps → LSTM(25 units) → Dropout(0.16) → Softmax(2)
- **LOG**: Logistic Regression with L2 regularization, 31 multi-period cumulative return features
- **RAF**: Random Forest, 1000 trees, max depth 20, 31 features
- **DNN**: Feedforward network (31→31→10→5→2), Dropout(0.5)

## Data
- Source: CRSP via WRDS
- Universe: S&P 500 historical constituents (survivor-bias free)
- Study periods: 23 × (750-day train + 250-day trade), non-overlapping trade windows
- LSTM input: 240-day rolling standardized return sequences
- Target: Binary label — whether stock outperforms cross-sectional median next day

## Key Results
| Model | Mean Accuracy |
|-------|--------------|
| LSTM  | 51.50%       |
| RAF   | 51.24%       |
| LOG   | 51.15%       |
| DNN   | 51.11%       |

LSTM outperforms all benchmarks, consistent with Fischer & Krauss (2017).

## Output Files
- `predictions/all_predictions.parquet`: Full prediction probabilities for all 23 periods
- `predictions/accuracy_by_period.png`: Accuracy comparison chart
- `predictions/training_diagnostics_period1.png`: LSTM training loss curve
- `predictions/hyperparameter_log.csv`: Model hyperparameter summary

## 1. Environment Setup
Import required libraries and configure device (MPS GPU acceleration on Apple Silicon).

In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import glob
import os
import warnings
warnings.filterwarnings('ignore')

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
train_files = sorted(glob.glob("output/train_samples_period_*.parquet"))
trade_files = sorted(glob.glob("output/trade_samples_period_*.parquet"))
print(f"Using device: {device}")
print(f"Found {len(train_files)} periods")

ModuleNotFoundError: No module named 'torch'

## 2. Load Data
Load the pre-processed train and trade sample files generated by the data pipeline (Part A).
Each study period contains 750-day training and 250-day trading windows.

In [ ]:
import os
import glob

# Load all period train and trade samples
data_dir = "output"

train_files = sorted(glob.glob(f"{data_dir}/train_samples_period_*.parquet"))
trade_files = sorted(glob.glob(f"{data_dir}/trade_samples_period_*.parquet"))

print(f"Found {len(train_files)} training periods")
print(f"Found {len(trade_files)} trading periods")

# Check data structure of first period
train_p1 = pd.read_parquet(train_files[0])
print("\nTraining data shape:", train_p1.shape)
print("Columns:", train_p1.columns.tolist())
print(train_p1.head(2))

## 3. Feature Construction - LSTM Sequences
Build 240-step rolling return sequences for LSTM input.
Only rows with `available_for_feature_generation=True` are used.
Standardization parameters (mean, std) are derived from training set only to avoid look-ahead bias.

In [ ]:
def build_sequences(df, seq_len=240):
    df = df.copy()
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values(['permno', 'date']).reset_index(drop=True)
    
    usable = df[df['available_for_feature_generation'] == True].copy()
    
    sequences = []
    labels = []
    meta = []
    
    for permno, group in usable.groupby('permno'):
        group = group.sort_values('date').reset_index(drop=True)
        ret_z = group['ret_z'].values
        label = group['label_t1'].values
        dates = group['date'].values
        
        for i in range(seq_len - 1, len(group)):
            seq = ret_z[i - seq_len + 1: i + 1]
            lbl = label[i]
            # Skip NaN labels
            if len(seq) == seq_len and not np.isnan(seq).any() and not np.isnan(lbl):
                sequences.append(seq)
                labels.append(int(lbl))
                meta.append({'date': dates[i], 'permno': permno})
    
    X = np.array(sequences, dtype=np.float32)
    y = np.array(labels, dtype=np.int64)
    meta_df = pd.DataFrame(meta)
    
    return X, y, meta_df

# Test on period 1
X_train, y_train, meta_train = build_sequences(train_p1)
print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"Label distribution: {np.bincount(y_train)}")

## 4. LSTM Model Architecture
Following Fischer & Krauss (2017):
- Input: 1 feature × 240 timesteps
- Hidden: 25 LSTM units with dropout = 0.16
- Output: 2 neurons + Softmax activation
- Optimizer: RMSprop
- Loss: Cross-Entropy
- Early stopping: patience = 10, max epochs = 1000, validation split = 80/20

In [ ]:
class LSTMModel(nn.Module):
    def __init__(self, input_size=1, hidden_size=25, dropout=0.16, num_classes=2):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            batch_first=True,
            dropout=0
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, num_classes)
    
    def forward(self, x):
        # x shape: (batch, seq_len, input_size)
        lstm_out, (h_n, c_n) = self.lstm(x)
        # Take last timestep output
        out = lstm_out[:, -1, :]
        out = self.dropout(out)
        out = self.fc(out)
        return out

# Test model
model = LSTMModel().to(device)
print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {total_params}")

## 5. Training Functions
Define training loop with early stopping and prediction functions for LSTM and benchmarks.

In [ ]:
class SequenceDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32).unsqueeze(-1)  # (N, 240, 1)
        self.y = torch.tensor(y, dtype=torch.long)
    
    def __len__(self):
        return len(self.y)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


def train_lstm(X_train, y_train, device, hidden_size=25, dropout=0.16,
               max_epochs=1000, patience=10, val_ratio=0.2, batch_size=512):
    
    # Train/val split (80/20)
    n = len(X_train)
    n_val = int(n * val_ratio)
    n_tr = n - n_val
    
    X_tr, X_val = X_train[:n_tr], X_train[n_tr:]
    y_tr, y_val = y_train[:n_tr], y_train[n_tr:]
    
    tr_loader = DataLoader(SequenceDataset(X_tr, y_tr), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(SequenceDataset(X_val, y_val), batch_size=batch_size, shuffle=False)
    
    model = LSTMModel(hidden_size=hidden_size, dropout=dropout).to(device)
    optimizer = torch.optim.RMSprop(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()
    
    best_val_loss = float('inf')
    best_weights = None
    epochs_no_improve = 0
    
    for epoch in range(max_epochs):
        # Train
        model.train()
        for X_batch, y_batch in tr_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            out = model(X_batch)
            loss = criterion(out, y_batch)
            loss.backward()
            optimizer.step()
        
        # Validate
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                out = model(X_batch)
                val_loss += criterion(out, y_batch).item()
        val_loss /= len(val_loader)
        
        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_weights = {k: v.clone() for k, v in model.state_dict().items()}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
        
        if epochs_no_improve >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break
    
    model.load_state_dict(best_weights)
    return model


def predict_proba_lstm(model, X, device, batch_size=512):
    model.eval()
    dataset = SequenceDataset(X, np.zeros(len(X), dtype=np.int64))
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    probs = []
    with torch.no_grad():
        for X_batch, _ in loader:
            X_batch = X_batch.to(device)
            out = model(X_batch)
            prob = torch.softmax(out, dim=1)[:, 1].cpu().numpy()
            probs.append(prob)
    return np.concatenate(probs)

## 6. Benchmark Feature Construction
Build multi-period cumulative return features for memory-free benchmark models.
Following Krauss et al. (2017): m ∈ {1,...,20, 40, 60,...,240} → 31 features total.

In [ ]:
def build_benchmark_features(df, train_df=None, seq_len=240):
    """
    Build multi-period cumulative return features for benchmark models.
    If train_df is provided, use it to extend history for trade period.
    """
    if train_df is not None:
        # Combine train tail + trade for feature construction
        df_combined = pd.concat([train_df, df], ignore_index=True)
    else:
        df_combined = df.copy()
    
    df_combined['date'] = pd.to_datetime(df_combined['date'])
    df_combined = df_combined.sort_values(['permno', 'date']).reset_index(drop=True)
    
    # Get trade dates
    df['date'] = pd.to_datetime(df['date'])
    trade_dates = set(df['date'].unique())
    
    periods = list(range(1, 21)) + list(range(40, 241, 20))
    
    features = []
    labels = []
    meta = []
    
    for permno, group in df_combined.groupby('permno'):
        group = group.sort_values('date').reset_index(drop=True)
        ret = group['ret'].values
        label = group['label_t1'].values
        dates = group['date'].values
        
        for i in range(max(periods), len(group)):
            if pd.Timestamp(dates[i]) not in trade_dates:
                continue
            if pd.isna(label[i]):
                continue
            row = []
            valid = True
            for m in periods:
                cum_ret = np.prod(1 + ret[i-m:i]) - 1
                if np.isnan(cum_ret):
                    valid = False
                    break
                row.append(cum_ret)
            if valid:
                features.append(row)
                labels.append(label[i])
                meta.append({'date': dates[i], 'permno': permno})
    
    X = np.array(features, dtype=np.float32)
    y = np.array(labels, dtype=np.int64)
    meta_df = pd.DataFrame(meta)
    return X, y, meta_df

## 7. Benchmark Models
- **Logistic Regression**: StandardScaler + L2 regularization
- **Random Forest**: 1000 trees, max depth 20

In [ ]:
from sklearn.preprocessing import StandardScaler

def train_logistic(X_train, y_train, X_test):
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    clf = LogisticRegression(max_iter=100, solver='lbfgs', C=1.0)
    clf.fit(X_train_scaled, y_train)
    probs = clf.predict_proba(X_test_scaled)[:, 1]
    return probs

def train_random_forest(X_train, y_train, X_test):
    clf = RandomForestClassifier(n_estimators=1000, max_depth=20, n_jobs=-1, random_state=42)
    clf.fit(X_train, y_train)
    probs = clf.predict_proba(X_test)[:, 1]
    return probs

# Train benchmarks on period 1
print("Training Logistic Regression...")
log_probs = train_logistic(X_bench_train, y_bench_train, X_bench_trade)
log_acc = accuracy_score(y_bench_trade, (log_probs >= 0.5).astype(int))
print(f"Period 1 LOG Accuracy: {log_acc:.4f}")

print("Training Random Forest...")
rf_probs = train_random_forest(X_bench_train, y_bench_train, X_bench_trade)
rf_acc = accuracy_score(y_bench_trade, (rf_probs >= 0.5).astype(int))
print(f"Period 1 RAF Accuracy: {rf_acc:.4f}")

## 8. DNN Benchmark
Feedforward neural network baseline following Krauss et al. (2017).
Architecture: FC(31→31→10→5→2) with Dropout(0.5) and Adam optimizer.

In [ ]:
class DNNModel(nn.Module):
    def __init__(self, input_size=31, num_classes=2):
        super(DNNModel, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, 31),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(31, 10),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(10, 5),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(5, num_classes)
        )
    
    def forward(self, x):
        return self.net(x)


def train_dnn(X_train, y_train, device, max_epochs=100, patience=10, 
              val_ratio=0.2, batch_size=512):
    
    n = len(X_train)
    n_val = int(n * val_ratio)
    n_tr = n - n_val
    
    X_tr, X_val = X_train[:n_tr], X_train[n_tr:]
    y_tr, y_val = y_train[:n_tr], y_train[n_tr:]
    
    # Standardize
    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_tr)
    X_val = scaler.transform(X_val)
    
    class TabularDataset(Dataset):
        def __init__(self, X, y):
            self.X = torch.tensor(X, dtype=torch.float32)
            self.y = torch.tensor(y, dtype=torch.long)
        def __len__(self): return len(self.y)
        def __getitem__(self, idx): return self.X[idx], self.y[idx]
    
    tr_loader = DataLoader(TabularDataset(X_tr, y_tr), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(TabularDataset(X_val, y_val), batch_size=batch_size, shuffle=False)
    
    model = DNNModel(input_size=X_train.shape[1]).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()
    
    best_val_loss = float('inf')
    best_weights = None
    epochs_no_improve = 0
    
    for epoch in range(max_epochs):
        model.train()
        for X_batch, y_batch in tr_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            loss = criterion(model(X_batch), y_batch)
            loss.backward()
            optimizer.step()
        
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                val_loss += criterion(model(X_batch), y_batch).item()
        val_loss /= len(val_loader)
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_weights = {k: v.clone() for k, v in model.state_dict().items()}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
        
        if epochs_no_improve >= patience:
            break
    
    model.load_state_dict(best_weights)
    return model, scaler


def predict_proba_dnn(model, scaler, X, device, batch_size=512):
    X_scaled = scaler.transform(X)
    X_tensor = torch.tensor(X_scaled, dtype=torch.float32)
    model.eval()
    probs = []
    with torch.no_grad():
        for i in range(0, len(X_tensor), batch_size):
            batch = X_tensor[i:i+batch_size].to(device)
            out = model(batch)
            prob = torch.softmax(out, dim=1)[:, 1].cpu().numpy()
            probs.append(prob)
    return np.concatenate(probs)

In [ ]:
def train_lstm_with_logging(X_train, y_train, device, hidden_size=25, dropout=0.16,
               max_epochs=1000, patience=10, val_ratio=0.2, batch_size=512):
    
    n = len(X_train)
    n_val = int(n * val_ratio)
    n_tr = n - n_val
    
    X_tr, X_val = X_train[:n_tr], X_train[n_tr:]
    y_tr, y_val = y_train[:n_tr], y_train[n_tr:]
    
    tr_loader = DataLoader(SequenceDataset(X_tr, y_tr), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(SequenceDataset(X_val, y_val), batch_size=batch_size, shuffle=False)
    
    model = LSTMModel(hidden_size=hidden_size, dropout=dropout).to(device)
    optimizer = torch.optim.RMSprop(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()
    
    best_val_loss = float('inf')
    best_weights = None
    epochs_no_improve = 0
    train_losses = []
    val_losses = []
    
    for epoch in range(max_epochs):
        # Train
        model.train()
        train_loss = 0
        for X_batch, y_batch in tr_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            out = model(X_batch)
            loss = criterion(out, y_batch)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        train_loss /= len(tr_loader)
        train_losses.append(train_loss)
        
        # Validate
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                out = model(X_batch)
                val_loss += criterion(out, y_batch).item()
        val_loss /= len(val_loader)
        val_losses.append(val_loss)
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_weights = {k: v.clone() for k, v in model.state_dict().items()}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
        
        if epochs_no_improve >= patience:
            break
    
    model.load_state_dict(best_weights)
    
    log = {
        'early_stopping_epoch': epoch + 1,
        'best_val_loss': best_val_loss,
        'train_losses': train_losses,
        'val_losses': val_losses
    }
    
    return model, log

# Run on period 1 as example for diagnostics plot
print("Training LSTM on period 1 with logging...")
train_p1 = pd.read_parquet(train_files[0])
X_tr, y_tr, _ = build_sequences(train_p1)
model_p1, log_p1 = train_lstm_with_logging(X_tr, y_tr, device)

print(f"Early stopping epoch: {log_p1['early_stopping_epoch']}")
print(f"Best val loss: {log_p1['best_val_loss']:.4f}")

In [ ]:
def predict_all_trade_days(model, train_df, trade_df, device, seq_len=240, batch_size=512):
    train_df = train_df.copy()
    trade_df = trade_df.copy()
    
    train_df['date'] = pd.to_datetime(train_df['date'])
    trade_df['date'] = pd.to_datetime(trade_df['date'])
    
    # Get trade dates as set of Timestamps
    trade_dates = set(pd.to_datetime(trade_df['date'].unique()))
    
    # Combine train + trade
    combined = pd.concat([train_df, trade_df], ignore_index=True)
    combined['date'] = pd.to_datetime(combined['date'])
    combined = combined.sort_values(['permno', 'date']).reset_index(drop=True)
    
    results = []
    
    for permno, group in combined.groupby('permno'):
        group = group.sort_values('date').reset_index(drop=True)
        ret_z = group['ret_z'].values
        dates = group['date'].values
        labels = group['label_t1'].values
        
        for i in range(seq_len - 1, len(group)):
            d = pd.Timestamp(dates[i])
            if d not in trade_dates:
                continue
            seq = ret_z[i - seq_len + 1: i + 1]
            if len(seq) == seq_len and not np.isnan(seq).any():
                lbl = labels[i]
                results.append({
                    'date': d,
                    'permno': permno,
                    'seq': seq,
                    'true_label': float(lbl) if not (lbl is None) else -1
                })
    
    if not results:
        return pd.DataFrame()
    
    X = np.array([r['seq'] for r in results], dtype=np.float32)
    probs = predict_proba_lstm(model, X, device, batch_size=batch_size)
    
    meta = pd.DataFrame([{
        'date': r['date'],
        'permno': r['permno'],
        'true_label': r['true_label']
    } for r in results])
    meta['lstm_prob'] = probs
    
    return meta

In [ ]:
daily_test = predict_all_trade_days(model_test, train_p1, trade_p1, device)
print(f"Rows: {len(daily_test)}, Unique dates: {daily_test['date'].nunique()}")

## 9. Daily Prediction Generation - All 23 Periods
Generate daily prediction probabilities for all models across 23 rolling study periods.
- LSTM: predictions_daily/lstm_all_periods.parquet
- LOG/RAF/DNN: predictions_daily/benchmark_all_periods.parquet

In [ ]:
# Test period 1 full pipeline
train_df = pd.read_parquet(train_files[0])
trade_df = pd.read_parquet(trade_files[0])

X_tr, y_tr, _ = build_sequences(train_df)
model = train_lstm(X_tr, y_tr, device)

daily_preds = predict_all_trade_days(model, train_df, trade_df, device)
daily_preds['period'] = 1

print(f"Period 1: {len(daily_preds)} rows, {daily_preds['date'].nunique()} unique dates")
print(daily_preds.head())

In [ ]:
import os
os.makedirs("predictions_daily", exist_ok=True)

all_daily_results = []

for i, (tr_file, td_file) in enumerate(zip(train_files, trade_files)):
    period = i + 1
    print(f"\n=== Period {period}/23 ===")
    
    train_df = pd.read_parquet(tr_file)
    trade_df = pd.read_parquet(td_file)
    
    # Train LSTM
    X_tr, y_tr, _ = build_sequences(train_df)
    model = train_lstm(X_tr, y_tr, device)
    
    # Generate predictions for every stock every day
    daily_preds = predict_all_trade_days(model, train_df, trade_df, device)
    daily_preds['period'] = period
    
    print(f"Daily predictions: {len(daily_preds)} rows, {daily_preds['date'].nunique()} unique dates")
    
    all_daily_results.append(daily_preds)
    daily_preds.to_parquet(f"predictions_daily/lstm_period_{period:02d}.parquet")

lstm_daily = pd.concat(all_daily_results, ignore_index=True)
lstm_daily.to_parquet("predictions_daily/lstm_all_periods.parquet")
print(f"\nDone! Total rows: {len(lstm_daily)}, Unique dates: {lstm_daily['date'].nunique()}")

In [ ]:
import os
os.makedirs("predictions_daily", exist_ok=True)

all_benchmark_results = []

for i, (tr_file, td_file) in enumerate(zip(train_files, trade_files)):
    period = i + 1
    print(f"\n=== Period {period}/23 ===")
    
    train_df = pd.read_parquet(tr_file)
    trade_df = pd.read_parquet(td_file)
    
    # Build benchmark features
    X_btr, y_btr, _ = build_benchmark_features(train_df)
    X_btd, y_btd, meta_btd = build_benchmark_features(trade_df, train_df=train_df)
    
    # Train LOG, RAF, DNN
    log_probs = train_logistic(X_btr, y_btr, X_btd)
    print(f"LOG done")
    
    rf_probs = train_random_forest(X_btr, y_btr, X_btd)
    print(f"RAF done")
    
    dnn_model, dnn_scaler = train_dnn(X_btr, y_btr, device)
    dnn_probs = predict_proba_dnn(dnn_model, dnn_scaler, X_btd, device)
    print(f"DNN done")
    
    meta_btd['log_prob'] = log_probs
    meta_btd['rf_prob'] = rf_probs
    meta_btd['dnn_prob'] = dnn_probs
    meta_btd['period'] = period
    
    all_benchmark_results.append(meta_btd[['date','permno','period','log_prob','rf_prob','dnn_prob']])
    print(f"Period {period}: {meta_btd['date'].nunique()} unique dates")

benchmark_daily = pd.concat(all_benchmark_results, ignore_index=True)
benchmark_daily.to_parquet("predictions_daily/benchmark_all_periods.parquet")
print(f"\nDone! Total rows: {len(benchmark_daily)}, Unique dates: {benchmark_daily['date'].nunique()}")

## 10. Benchmark Comparison Table
Aggregate out-of-sample accuracy across all 23 periods for each model.

In [ ]:
# Final benchmark comparison table across all periods
final = pd.read_parquet("predictions/all_predictions.parquet")

summary = []
for period in range(1, 24):
    p = final[final['period'] == period]
    p_valid = p.dropna(subset=['log_prob', 'rf_prob', 'dnn_prob'])
    
    lstm_acc = accuracy_score(p['true_label'], (p['lstm_prob'] >= 0.5).astype(int))
    log_acc = accuracy_score(p_valid['true_label'], (p_valid['log_prob'] >= 0.5).astype(int))
    rf_acc = accuracy_score(p_valid['true_label'], (p_valid['rf_prob'] >= 0.5).astype(int))
    dnn_acc = accuracy_score(p_valid['true_label'], (p_valid['dnn_prob'] >= 0.5).astype(int))
    
    summary.append({'Period': period, 'LSTM': lstm_acc, 'LOG': log_acc, 'RAF': rf_acc, 'DNN': dnn_acc})

summary_df = pd.DataFrame(summary)
print(summary_df.round(4).to_string(index=False))
print(f"\nMean Accuracy across all periods:")
print(f"LSTM: {summary_df['LSTM'].mean():.4f}")
print(f"LOG:  {summary_df['LOG'].mean():.4f}")
print(f"RAF:  {summary_df['RAF'].mean():.4f}")
print(f"DNN:  {summary_df['DNN'].mean():.4f}")

## 11. Accuracy by Period - Visualization
Plot out-of-sample prediction accuracy across study periods.
Note the declining trend in later periods, consistent with increasing market efficiency over time.

In [ ]:
import matplotlib.pyplot as plt

# Plot accuracy by period for all models
fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(summary_df['Period'], summary_df['LSTM'], marker='o', label='LSTM', linewidth=2)
ax.plot(summary_df['Period'], summary_df['RAF'], marker='s', label='RAF', linewidth=2)
ax.plot(summary_df['Period'], summary_df['LOG'], marker='^', label='LOG', linewidth=2)
ax.plot(summary_df['Period'], summary_df['DNN'], marker='D', label='DNN', linewidth=2)
ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.7, label='Random (50%)')

ax.set_xlabel('Study Period')
ax.set_ylabel('Accuracy')
ax.set_title('Out-of-Sample Prediction Accuracy by Period (k=all stocks)')
ax.legend()
ax.set_xticks(range(1, 24))
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('predictions/accuracy_by_period.png', dpi=150)
plt.show()
print("Saved.")

## 12. Training Diagnostics
Re-train LSTM on Period 1 with full loss logging to visualize training behavior.
This serves as a representative example of the early stopping mechanism.

In [ ]:
# Training diagnostics - loss curve for Period 1
fig, ax = plt.subplots(figsize=(8, 4))

epochs = range(1, len(log_p1['train_losses']) + 1)
ax.plot(epochs, log_p1['train_losses'], label='Train Loss', linewidth=2)
ax.plot(epochs, log_p1['val_losses'], label='Validation Loss', linewidth=2)
ax.axvline(x=log_p1['early_stopping_epoch'] - 10, color='red', 
           linestyle='--', alpha=0.7, label=f"Early Stop (Epoch {log_p1['early_stopping_epoch']})")

ax.set_xlabel('Epoch')
ax.set_ylabel('Cross-Entropy Loss')
ax.set_title('LSTM Training Diagnostics - Period 1 (Representative Example)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('predictions/training_diagnostics_period1.png', dpi=150)
plt.show()
print("Saved.")

## 13. Hyperparameter Log
Summary of model configurations and key hyperparameters for all four models.

In [ ]:
# Hyperparameter summary table
hparam_log = {
    'Model': ['LSTM', 'LOG', 'RAF', 'DNN'],
    'Architecture': [
        'LSTM(25 units) + Dropout(0.16) + Softmax',
        'L2 regularization, LBFGS solver',
        '1000 trees, max_depth=20',
        'FC(31→31→10→5→2) + Dropout(0.5)'
    ],
    'Optimizer': ['RMSprop', 'LBFGS', 'N/A', 'Adam'],
    'Input Features': [
        '240-step standardized return sequence',
        '31 multi-period cumulative returns',
        '31 multi-period cumulative returns',
        '31 multi-period cumulative returns'
    ],
    'Early Stopping': [
        f'patience=10, epoch={log_p1["early_stopping_epoch"]} (Period 1)',
        'N/A',
        'N/A',
        'patience=10'
    ],
    'Mean Accuracy': [
        f'{summary_df["LSTM"].mean():.4f}',
        f'{summary_df["LOG"].mean():.4f}',
        f'{summary_df["RAF"].mean():.4f}',
        f'{summary_df["DNN"].mean():.4f}'
    ]
}

hparam_df = pd.DataFrame(hparam_log)
print(hparam_df.to_string(index=False))
hparam_df.to_csv('predictions/hyperparameter_log.csv', index=False)
print("\nSaved to predictions/hyperparameter_log.csv")